<a href="https://colab.research.google.com/github/ntpism17/myproj/blob/main/thai_election_ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thai Election OCR — Hackathon Solution

**Goal**: Extract vote counts from scanned Thai election result documents (Form สส.6/1)  
**Metric**: Mean Levenshtein Distance (lower = better)  
**Approach**: Azure Document Intelligence (prebuilt-layout) + Fuzzy Party Matching

## Pipeline Overview
```
Images (PNG) → Azure Document Intelligence → Table Extraction
           → Party Name Matching (exact/substring/fuzzy)
           → submission.csv
```

## 1. Setup & Install

In [ ]:
!pip install requests pandas -q

In [ ]:
import json, re, sys, time, base64, difflib, requests
import pandas as pd
from pathlib import Path

print('Libraries loaded successfully')

## 2. Configuration

In [ ]:
# ── Azure Document Intelligence ──
ENDPOINT = "https://ntp-docint.cognitiveservices.azure.com"
API_KEY  = "YOUR_API_KEY_HERE"
API_VER  = "2024-11-30"

# ── Paths (adjust to your environment) ──
DATA_DIR  = Path("data")
IMG_DIR   = DATA_DIR / "images"
TEMPLATE  = DATA_DIR / "submission_template.csv"
OUTPUT    = Path("submission.csv")
CACHE     = Path("ocr_cache.json")

PAGE_DELAY    = 2
POLL_INTERVAL = 2
RETRY_DELAYS  = [5, 10, 20]

# Thai → Arabic digit mapping
THAI_DIGIT_MAP = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
print('Config set')

## 3. Data Cleaning Helpers

In [ ]:
def thai_to_arabic(s: str) -> str:
    """Convert Thai digits to Arabic: '๓๔,๑๖๗' → '34,167'"""
    return s.translate(THAI_DIGIT_MAP)


def clean_votes(raw: str) -> str:
    """Extract numeric vote count from raw OCR text.

    Examples:
        '๓๔,๑๖๗ (สามหมื่น...)' → '34167'
        '14,813'               → '14813'
        '2569' (Buddhist year) → '0'
    """
    s = thai_to_arabic(str(raw))
    s = s.split("(")[0]
    digits = re.sub(r"[^\d]", "", s)
    if not digits:
        return "0"
    val = int(digits)
    # Filter Buddhist Era years (2400–2700) — not vote counts
    if 2400 <= val <= 2700 and len(digits) == 4:
        return "0"
    return digits


def strip_party_prefix(name: str) -> str:
    """Remove 'พรรค' prefix: 'พรรคประชาชน' → 'ประชาชน'"""
    name = name.strip()
    if name.startswith("พรรค"):
        name = name[4:]
    return name.strip()


def normalize(name: str) -> str:
    """Normalize party name for comparison (strip prefix + whitespace)."""
    if not isinstance(name, str):
        return ""
    s = strip_party_prefix(name)
    return s.strip().replace(" ", "").replace("\u200b", "")


# Quick test
assert clean_votes("๓๔,๑๖๗ (สามหมื่น)") == "34167"
assert clean_votes("2569") == "0"
assert strip_party_prefix("พรรคประชาชน") == "ประชาชน"
print('Data cleaning helpers OK')

## 4. Party Name Classification

In [ ]:
def is_candidate_name(text: str) -> bool:
    """Return True if text looks like a candidate name (นาย/นาง/ดร. etc.)"""
    PREFIXES = ["นาย", "นาง", "น.ส.", "ว่าที่", "พัน", "ร้อย", "พล",
                "จ่า", "สิบ", "ดร.", "ศ.", "ผศ.", "รศ.", "ดาบ"]
    return any(text.strip().startswith(p) for p in PREFIXES)


def is_party_name(text: str) -> bool:
    """Return True if text looks like a Thai political party name."""
    t = text.strip()
    if not t or len(t) < 2:
        return False
    if is_candidate_name(t):
        return False
    # Must be mostly Thai characters
    thai_count = sum(1 for c in t if '\u0e00' <= c <= '\u0e7f')
    if thai_count < len(t) * 0.5:
        return False
    # Must not contain digits
    if re.search(r"\d", thai_to_arabic(t)):
        return False
    SKIP = ["รวมคะแนน", "รวมทั้งสิ้น", "ประกาศ", "กรรมการ",
            "หมายเลข", "สังกัด", "ได้คะแนน", "ลงชื่อ"]
    return not any(kw in t for kw in SKIP)


print('Party name classifier OK')

## 5. Azure Document Intelligence API

In [ ]:
def img_b64(path: Path) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def analyze_image(image_path: Path) -> dict | None:
    """
    Submit image to Azure Document Intelligence (prebuilt-layout).
    Returns analyzeResult dict or None on failure.

    Why prebuilt-layout?
    - Returns tables with explicit rowIndex/columnIndex per cell
    - Handles merged cells (rowspan/colspan) correctly
    - No prompt engineering needed
    - Fast: ~4-6 seconds per page
    """
    url = (f"{ENDPOINT}/documentintelligence/documentModels/"
           f"prebuilt-layout:analyze?api-version={API_VER}")
    headers = {
        "Ocp-Apim-Subscription-Key": API_KEY,
        "Content-Type": "application/json"
    }

    for attempt, wait in enumerate([0] + RETRY_DELAYS):
        if wait:
            print(f"  Retry in {wait}s...")
            time.sleep(wait)
        try:
            r = requests.post(url, headers=headers,
                              json={"base64Source": img_b64(image_path)},
                              timeout=60)
            if r.status_code == 429:
                continue
            if r.status_code != 202:
                print(f"  Submit failed: HTTP {r.status_code}")
                return None

            result_url = r.headers.get("Operation-Location")
            # Poll until complete
            for _ in range(30):
                time.sleep(POLL_INTERVAL)
                pr = requests.get(result_url,
                                  headers={"Ocp-Apim-Subscription-Key": API_KEY},
                                  timeout=30)
                result = pr.json()
                if result.get("status") == "succeeded":
                    return result.get("analyzeResult", {})
                elif result.get("status") == "failed":
                    return None
        except Exception as e:
            print(f"  Error: {e}")
    return None


print('Azure Document Intelligence API function ready')

## 6. Table Parsing

The election document table has columns:  
`[หมายเลข | ชื่อ-สกุล | สังกัดพรรค | ได้คะแนน | สถานีที่1 | สถานีที่2 ...]`

Strategy:
1. Find **votes column** = column with most large-number cells (2-7 digits)
2. Find **party column** = column with most valid party name cells (before votes col)
3. Extract pairs row by row

In [ ]:
def parse_tables(analyze_result: dict) -> list[dict]:
    """
    Extract [{party, votes}] from Document Intelligence table results.
    Uses rowIndex/columnIndex for accurate cell positioning.
    """
    if not analyze_result:
        return []

    tables = analyze_result.get("tables", [])
    if not tables:
        return []

    # Use largest table (most cells)
    main_table = max(tables, key=lambda t: t.get("rowCount", 0) * t.get("columnCount", 0))
    row_count = main_table.get("rowCount", 0)
    col_count = main_table.get("columnCount", 0)

    if row_count < 3 or col_count < 3:
        return []

    # Build grid with rowSpan/colSpan support
    grid = [[""]*col_count for _ in range(row_count)]
    for cell in main_table.get("cells", []):
        r, c = cell.get("rowIndex", 0), cell.get("columnIndex", 0)
        content = cell.get("content", "").strip()
        for dr in range(cell.get("rowSpan", 1)):
            for dc in range(cell.get("colSpan", 1)):
                if r+dr < row_count and c+dc < col_count:
                    grid[r+dr][c+dc] = content

    # Score each column
    party_scores = [0] * col_count
    votes_scores = [0] * col_count
    for row in range(1, row_count):
        for col in range(col_count):
            text = grid[row][col]
            if not text:
                continue
            digits = re.sub(r"[^\d]", "", thai_to_arabic(text).split("(")[0])
            if digits and 2 <= len(digits) <= 7:
                val = int(digits)
                if not (2400 <= val <= 2700 and len(digits) == 4):
                    votes_scores[col] += 1
            elif is_party_name(text):
                party_scores[col] += 1

    # Pick best columns
    vote_candidates = sorted([(votes_scores[c], c) for c in range(1, col_count)], reverse=True)
    if not vote_candidates or vote_candidates[0][0] == 0:
        return []
    votes_col = vote_candidates[0][1]

    party_candidates = sorted([(party_scores[c], c) for c in range(1, votes_col)], reverse=True)
    party_col = party_candidates[0][1] if party_candidates and party_candidates[0][0] > 0 else votes_col - 1

    # Extract party-votes pairs
    SKIP_KW = ["รวมคะแนน", "รวมทั้งสิ้น", "ประกาศ", "กรรมการ", "ลงชื่อ"]
    rows, seen = [], set()
    for row in range(1, row_count):
        party_text = grid[row][party_col]
        votes_text = grid[row][votes_col]
        if not party_text or not votes_text:
            continue
        if any(kw in party_text for kw in SKIP_KW) or not is_party_name(party_text):
            continue
        votes = clean_votes(votes_text)
        if votes == "0":
            continue
        party = strip_party_prefix(party_text)
        if party and party not in seen:
            rows.append({"party": party, "votes": votes})
            seen.add(party)
    return rows


print('Table parser ready')

## 7. Document Processing with Cache

In [ ]:
def get_pages(doc_id: str) -> list[Path]:
    """Get all page image paths for a document."""
    pages = []
    if (IMG_DIR / f"{doc_id}.png").exists():
        pages.append(IMG_DIR / f"{doc_id}.png")
    i = 2
    while (IMG_DIR / f"{doc_id}_page{i}.png").exists():
        pages.append(IMG_DIR / f"{doc_id}_page{i}.png")
        i += 1
    return pages


def load_cache() -> dict:
    if CACHE.exists():
        with open(CACHE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_cache(cache: dict):
    with open(CACHE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def process_doc(doc_id: str, cache: dict) -> dict[str, str]:
    """Process one document: OCR all pages, extract party-votes pairs."""
    if doc_id in cache and cache[doc_id]:
        return cache[doc_id]

    pages = get_pages(doc_id)
    if not pages:
        cache[doc_id] = {}
        return {}

    # Skip page 1 for multi-page docs
    table_pages = pages[1:] if len(pages) > 1 else pages
    result: dict[str, str] = {}

    for i, page in enumerate(table_pages):
        print(f"  Page {i+2}/{len(pages)}: {page.name}")
        analyze = analyze_image(page)
        if analyze:
            rows = parse_tables(analyze)
            for r in rows:
                if r["party"] and r["votes"] != "0":
                    result[r["party"]] = r["votes"]
            print(f"    → {len(rows)} parties found")
        time.sleep(PAGE_DELAY)

    # Fallback: try page 1 if nothing found
    if not result and pages:
        print("  Retrying with page 1...")
        analyze = analyze_image(pages[0])
        if analyze:
            rows = parse_tables(analyze)
            for r in rows:
                result[r["party"]] = r["votes"]
        time.sleep(PAGE_DELAY)

    cache[doc_id] = result
    save_cache(cache)
    return result


print('Document processing functions ready')

## 8. Fuzzy Party Name Matching

Template party names may differ slightly from OCR output.  
We use 3-tier matching: **Exact → Substring → Fuzzy (difflib)**

In [ ]:
def match_votes(party_name: str, ocr_map: dict[str, str]) -> str:
    """
    Match template party name to OCR result.
    3-tier strategy:
      1. Exact match (after normalizing whitespace & 'พรรค' prefix)
      2. Substring match
      3. Fuzzy match (difflib, cutoff=0.7)
    Returns vote string or '0' if no match.
    """
    if not isinstance(party_name, str):
        return "0"

    norm_target = normalize(party_name)

    # Tier 1: Exact
    for k, v in ocr_map.items():
        if normalize(k) == norm_target:
            return v

    # Tier 2: Substring
    for k, v in ocr_map.items():
        nk = normalize(k)
        if norm_target in nk or nk in norm_target:
            return v

    # Tier 3: Fuzzy
    keys = list(ocr_map.keys())
    norm_keys = [normalize(k) for k in keys]
    matches = difflib.get_close_matches(norm_target, norm_keys, n=1, cutoff=0.7)
    if matches:
        return ocr_map[keys[norm_keys.index(matches[0])]]

    return "0"


# Test
test_map = {"ประชาชน": "34167", "ภูมิใจไทย": "14368"}
assert match_votes("พรรคประชาชน", test_map) == "34167"
assert match_votes("ภูมิใจ", test_map) == "14368"
print('Party matching OK')

## 9. Run Full Pipeline

In [ ]:
print('Loading template...')
df = pd.read_csv(TEMPLATE)
print(f'  {len(df)} rows, {df["doc_id"].nunique()} documents')

cache = load_cache()
print(f'  Cache: {len(cache)} docs already processed')

doc_ids = df["doc_id"].unique().tolist()
votes_map: dict[str, str] = {}
total = len(doc_ids)

for idx, doc_id in enumerate(doc_ids, 1):
    group = df[df["doc_id"] == doc_id]
    cached = doc_id in cache and bool(cache[doc_id])
    print(f'[{idx:3d}/{total}] {doc_id} ({len(group)} rows)' + (' [cached]' if cached else ''))

    ocr_map = process_doc(doc_id, cache)

    for _, row in group.iterrows():
        votes_map[row["id"]] = match_votes(row["party_name"], ocr_map)

print('\nPipeline complete!')

## 10. Build submission.csv

In [ ]:
submission = df[["id"]].copy()
submission["votes"] = submission["id"].map(votes_map).fillna("0")
submission.to_csv(OUTPUT, index=False)

matched = (submission["votes"] != "0").sum()
print(f'Total rows:  {len(submission)}')
print(f'Matched:     {matched} ({100*matched/len(submission):.1f}%)')
print(f'Unmatched:   {len(submission)-matched} (will be "0")')
print(f'\nSaved → {OUTPUT}')
submission.head(10)

## 11. Results Summary

In [ ]:
import matplotlib
matplotlib.use('Agg')  # for Colab
import matplotlib.pyplot as plt

# Vote distribution
votes_numeric = pd.to_numeric(submission['votes'], errors='coerce').dropna()
votes_nonzero = votes_numeric[votes_numeric > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Matched vs Unmatched
axes[0].pie([matched, len(submission)-matched],
            labels=['Matched', 'Unmatched (0)'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Row Match Rate')

# Vote distribution
axes[1].hist(votes_nonzero, bins=50, color='#3498db', edgecolor='white')
axes[1].set_title('Distribution of Vote Counts')
axes[1].set_xlabel('Votes')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('results_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Median votes (non-zero): {votes_nonzero.median():.0f}')
print(f'Max votes: {votes_nonzero.max():.0f}')
print(f'Min votes (non-zero): {votes_nonzero.min():.0f}')

## Approach Summary

| Step | Method | Why |
|------|--------|-----|
| OCR | Azure Document Intelligence `prebuilt-layout` | Extracts tables with row/col index → no rowspan bugs |
| Thai digits | `str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")` | Convert Thai to Arabic numerals |
| Vote cleaning | Regex + Buddhist year filter | Remove text, filter year values |
| Party matching | Exact → Substring → Fuzzy (difflib 0.7) | Handle OCR typos & 'พรรค' prefix |
| Column detection | Score-based (votes_score, party_score) | Auto-detect correct table columns |
| Cache | JSON file per pipeline | Resume-safe, no duplicate API calls |